In [1]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from pymongo import MongoClient
import yaml
from bson import json_util
from pydantic import BaseModel
from typing import Dict


In [2]:
class BackUpConfig(BaseModel):
    db: str
    url: str
    backup_folder: str
    collection: Dict[str, str]
    year: str


def load_config(path: str) -> BackUpConfig:
    with open(path, "r") as f:
        raw = yaml.safe_load(f)

    return BackUpConfig(
            db=raw["mongo"]["db"],
            url=raw["mongo"]["url"],
            backup_folder=raw["mongo"]["backup_folder"],
            collection=raw["mongo"]["collection"],
            year=raw["season"]["year"]

        )

events_config = load_config("../config/config.yaml")


In [4]:

backup_root = Path(events_config.backup_folder)
backup_root.mkdir(parents=True, exist_ok=True)

client = MongoClient(events_config.url)
db = client[events_config.db]

season = events_config.year
collections = {
    "schedule": events_config.collection["collection_schedule"],
    "raw_events": events_config.collection["collection_raw_events"],
}

for name, coll_name in collections.items():
    coll = db[coll_name]
    docs = list(coll.find({'season': season}))

    if not docs:
        print(f"No records found for season {season} in {name}")
        continue

    out_json_name = f"{season}_{name}.json"
    out_zip = backup_root / f"{season}_{name}.zip"

    json_text = json_util.dumps(docs, indent=2)

    with ZipFile(out_zip, "w", compression=ZIP_DEFLATED) as zf:
        zf.writestr(out_json_name, json_text.encode("utf-8"))

    print(f"Season: {season} -> Saved {len(docs)} documents for {name} and zipped to {out_zip}")

Season: 2025-2026 -> Saved 279 documents for schedule and zipped to /Users/mario_omescu/Library/Mobile Documents/com~apple~CloudDocs/Sports Analytics/WhoScored Events Data/mongo_backup/2025-2026_schedule.zip
Season: 2025-2026 -> Saved 436885 documents for raw_events and zipped to /Users/mario_omescu/Library/Mobile Documents/com~apple~CloudDocs/Sports Analytics/WhoScored Events Data/mongo_backup/2025-2026_raw_events.zip
